# 工具書05：論文壓縮主張與來源稽核

此頁準確率均為論文引用，不是本專案重現。

## 0. 我們真正要重現什麼？

**本節目標**：將貢獻拆成參數比例和準確率差距。

**為什麼需要這一步**：之前610參數小模型的生成器更大，沒有測到壓縮主張。

**輸入／輸出與 shape**：論文Tables1–3 → M、P_mapping、P_quantum、accuracy。

**關鍵原理與程式定位**：ratio=(mapping+quantum)/M；生成M個weights不表示M個獨立可訓練參數。

**操作步驟**：執行環境cell，再閱讀固定v2來源。此notebook不訓練、不下載MNIST。

**預期觀察**：明確區分published與本機observed。

**結果判讀與常見錯誤**：把最小參數數量與最大accuracy配在一起會虛構trade-off。

**理解問題**：前面小模型失敗能否否定論文壓縮？

<details><summary>提示與解答</summary>

不能；既沒有壓縮比，也沒有對齊論文資料/電路/模型。

</details>


In [ ]:
import os, sys, json, time
from pathlib import Path
import torch, pennylane as qml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from torch.utils.data import DataLoader
from qt_eval.config import load_config, validate_config
from qt_eval.data import load_digits_splits
from qt_eval.models import make_model, ClassicalMLP, functional_mlp
from qt_eval.quantum import QuantumTrain, make_qnode, basis_codes, probability_features
from qt_eval.diagnostics import check_training_step
from qt_eval.experiment import run_experiment, evaluate
from qt_eval.reporting import summarize_runs, write_report
from qt_eval.artifacts import SOURCE_COMMIT

# 教學預設 smoke；只有準備好正式預算後才手動改成 full。
MODE = os.environ.get('QT_EVAL_MODE', 'smoke')
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '請由專案根目錄或 notebooks 目錄啟動'
OUTPUT = Path(os.environ.get('QT_EVAL_OUTPUT', str(ROOT/'results'/'notebooks')))
CONFIG = load_config(ROOT/'configs'/('smoke.json' if MODE=='smoke' else 'depth_study.json'))
assert CONFIG['mode'] == MODE
torch.set_num_threads(CONFIG['torch_threads'])
torch.manual_seed(CONFIG['seed'])
print('MODE:', MODE, '| device: cpu | dtype: float64')
if MODE == 'smoke':
    print('工程檢查：小資料、短訓練，不可用來判斷模型研究表現。')

## 1. 計數與取捨表

**本節目標**：用程式核對paper宣稱。

**為什麼需要這一步**：參數口徑要包括mapping，不能只計量子角度。

**輸入／輸出與 shape**：MNIST mapping14→4→20→4→1=249；13×L量子參數。

**關鍵原理與程式定位**：QT16=249+208=457，QT106=249+1378=1627；準確率引用而非本機實測。

**操作步驟**：列印參數比例與accuracy gap，找出最小參數和較高準確率各是哪一列。

**預期觀察**：QT16的MNIST accuracy81.08%，不是QT106的93.81%。

**結果判讀與常見錯誤**：少參數不自动等於少總記憶體、快訓練；推論若展開完整weights仍有M個數值。

**理解問題**：CIFAR10較接近baseline的是哪個設定？

<details><summary>提示與解答</summary>

QT323：23258/285226≈8.15%，60.69%對62.50%，差1.81pp；需1000epochs，非短測結果。

</details>


In [ ]:
from qt_eval.paper_audit import compression_rows,public_notebook_count
claims=pd.DataFrame(compression_rows())
display(claims[['dataset','blocks','target_parameters','qt_parameters','parameter_ratio','published_qt_accuracy','accuracy_gap_pp']])
assert set(claims.evidence_type)=={'published_not_reproduced'}

## 2. 官方範例也需要版本對齊

**本節目標**：辨認論文與公開notebook差異。

**為什麼需要這一步**：同一repository可包含後續變體，不能只憑檔名L16認定對應表格。

**輸入／輸出與 shape**：RY/CNOT16blocks→208角度；U3/CU3版本→1248角度。

**關鍵原理與程式定位**：U3與CU3各3角度，每block13+13個gate：6×13×16；加mapping249為1497。

**操作步驟**：讀差異表，列出仍需作者或固定程式確認的設定。

**預期觀察**：能指出gate、mapping activation、機率處理與test評估不一致。

**結果判讀與常見錯誤**：目前只做靜態稽核，沒有執行TorchQuantum原notebook；數值結果仍未重現。

**理解問題**：換成main分支就能修正嗎？

<details><summary>提示與解答</summary>

此次查核main和chenyu_dev該MNIST檔案blob相同，不能解決差異。

</details>


In [ ]:
print('Public notebook static parameter count:',public_notebook_count())
display(Markdown((ROOT/'docs/paper-reproduction-audit.md').read_text(encoding='utf-8')))

## 3. 開始訓練前的證據門檻

**本節目標**：避免再以不同方法回答原論文問題。

**為什麼需要這一步**：精確設定有缺口，應先標示，而不是用常見寫法偷偷補齊。

**輸入／輸出與 shape**：來源矩陣 → 選定重現或有假設的重建協定。

**關鍵原理與程式定位**：先MNIST6690/13qubits，驗證CNN等價、mapping249、壓縮比例、梯度及匯出純CNN一致性，再測試執行成本。

**操作步驟**：核對未決電路初态/排序、mapping nonlinear位置、機率處理、初始化及評估；取得對應程式或明確採用重建假設後才長跑。

**預期觀察**：此階段得到可信audit，而不是新的accuracy。

**結果判讀與常見錯誤**：50epochs全MNIST與19qubits深CIFAR是不同預算；smoke只能證明工程流程。

**理解問題**：重建時使用了合理假設，可稱精確重現嗎？

<details><summary>提示與解答</summary>

不可以；應列出假設，結果標成paper-guided reconstruction。

</details>


In [ ]:
print('Status: source audit complete; exact numerical reproduction unresolved.')
print('MNIST target:',6690,'QT16:',457,'ratio:',457/6690)
print('Next: resolve paper/code mismatches before full training.')